# Суммаризатор текстовых файлов

Загружает текстовый файл и строит краткое изложение через **gpt-4.1-mini**.

## Установка зависимостей

In [2]:
%pip install openai python-dotenv httpx langfuse --quiet


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


## Инициализация клиента и Langfuse

In [3]:
import os
import httpx
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv(override=True)

http_client = httpx.Client(verify=False, timeout=60.0)

client = OpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url=os.getenv('OPENAI_BASE_URL'),
    timeout=60.0,
    max_retries=2,
    http_client=http_client,
)

MODEL = "gpt-4.1-mini"

resp = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "ping"}],
    max_tokens=5,
)
print(f"✅ OpenAI подключение успешно. Модель: {resp.model}")

✅ OpenAI подключение успешно. Модель: gpt-4.1-mini-2025-04-14


In [5]:
from langfuse import Langfuse, observe  # v4: observe импортируется напрямую, decorators убраны

langfuse = Langfuse(
    public_key=os.getenv('LANGFUSE_PUBLIC_KEY'),
    secret_key=os.getenv('LANGFUSE_SECRET_KEY'),
    host=os.getenv('LANGFUSE_BASE_URL'),
)

langfuse.auth_check()
print("✅ Langfuse подключение успешно.")

✅ Langfuse подключение успешно.


## Реализация суммаризатора

Если текст длиннее `CHUNK_SIZE` символов — он автоматически нарезается на чанки по абзацам,
каждый суммаризируется с контекстом предыдущих, затем итоговые фрагменты объединяются.

In [24]:
import time

CHUNK_SIZE = 12_000   # символов в одном чанке
SUMMARY_LANG = "русском"  # или "английском"

# Стоимость gpt-4.1-mini за 1M токенов (USD)
COST_PER_1M_INPUT  = 0.40
COST_PER_1M_OUTPUT = 1.60

SYSTEM_PROMPT = (
    f"Ты — помощник, который кратко и точно пересказывает текст на {SUMMARY_LANG} языке. "
    "Сохраняй ключевые факты, имена и выводы. Не добавляй ничего лишнего."
)


@observe(as_type="generation")
def summarize_chunk(chunk: str, extra_context: str = "", chunk_index: int = 0) -> str:
    """Суммаризирует один фрагмент текста."""
    user_content = chunk
    if extra_context:
        user_content = (
            f"[Контекст предыдущих частей]\n{extra_context}\n\n"
            f"[Текущий фрагмент]\n{chunk}"
        )
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": f"Кратко изложи следующий текст:\n\n{user_content}"},
    ]

    t0 = time.perf_counter()
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        temperature=0.3,
        max_tokens=1024,
    )
    elapsed = time.perf_counter() - t0

    in_tokens = response.usage.prompt_tokens
    out_tokens = response.usage.completion_tokens
    result = response.choices[0].message.content.strip()

    langfuse.update_current_generation(
        name=f"chunk_{chunk_index}",
        input=messages,
        output=result,
        model=MODEL,
        usage_details={"input": in_tokens, "output": out_tokens},
        cost_details={
            "input": round(in_tokens  * COST_PER_1M_INPUT  / 1_000_000, 6),
            "output": round(out_tokens * COST_PER_1M_OUTPUT / 1_000_000, 6),
        },
        metadata={
            "chunk_index": chunk_index,
            "input_chars": len(user_content),
            "output_chars": len(result),
            "latency_sec": round(elapsed, 3),
        },
    )
    return result


@observe(name="summarize_pipeline")
def summarize_text(text: str, file_path: str = "", chunk_size: int = CHUNK_SIZE) -> str:
    """Суммаризирует текст, при необходимости разбивая на чанки."""
    langfuse.update_current_span(
        input=text,
        metadata={
            "file": file_path,
            "source_chars": len(text),
            "source_words": len(text.split()),
        },
    )

    langfuse.create_event(
        name="file_loaded",
        input={"file": file_path, "chars": len(text), "words": len(text.split())},
    )

    t0 = time.perf_counter()

    if len(text) <= chunk_size:
        print("Текст помещается в один запрос.")
        langfuse.create_event(name="single_chunk", input={"chars": len(text)})
        result = summarize_chunk(text, chunk_index=0)
    else:
        chunks, current = [], ""
        for paragraph in text.split("\n"):
            if len(current) + len(paragraph) + 1 > chunk_size and current:
                chunks.append(current.strip())
                current = ""
            current += paragraph + "\n"
        if current.strip():
            chunks.append(current.strip())

        langfuse.create_event(
            name="chunks_ready",
            input={"count": len(chunks), "avg_chars": len(text) // len(chunks)},
        )
        print(f"Текст разбит на {len(chunks)} частей.")

        partial_summaries = []
        for i, chunk in enumerate(chunks, 1):
            print(f"  Обрабатываю часть {i}/{len(chunks)}...")
            context = " ".join(partial_summaries[-2:])
            partial_summaries.append(summarize_chunk(chunk, extra_context=context, chunk_index=i))
            langfuse.create_event(
                name="chunk_done",
                input={"chunk_index": i, "total": len(chunks)},
            )

        if len(partial_summaries) > 1:
            print("Финальное объединение...")
            langfuse.create_event(name="merging_chunks", input={"parts": len(partial_summaries)})
            combined = "\n\n".join(f"Часть {i+1}:\n{s}" for i, s in enumerate(partial_summaries))
            result = summarize_chunk(combined, chunk_index=0)
        else:
            result = partial_summaries[0]

    total_sec = time.perf_counter() - t0

    langfuse.create_event(
        name="summarization_done",
        input={"summary_chars": len(result), "summary_words": len(result.split()), "total_sec": round(total_sec, 3)},
    )

    langfuse.update_current_span(
        output=result,
        metadata={
            "file": file_path,
            "source_chars": len(text),
            "source_words": len(text.split()),
            "summary_chars": len(result),
            "summary_words": len(result.split()),
            "compression": round(len(result) / len(text), 3),
            "total_sec": round(total_sec, 3),
        },
    )
    langfuse.set_current_trace_io(input=text, output=result)
    return result

## Выбор файла для суммаризации

In [25]:
FILE_PATH = "sample.txt"  # ← укажите путь к вашему файлу

with open(FILE_PATH, encoding="utf-8") as f:
    text = f.read()

print(f"📄 Файл загружен: {FILE_PATH}")
print(f"   Символов: {len(text):,}")
print(f"   Слов:     {len(text.split()):,}")
print()
print("--- Начало файла (первые 500 символов) ---")
print(text[:500])

📄 Файл загружен: sample.txt
   Символов: 63,307
   Слов:     7,850

--- Начало файла (первые 500 символов) ---
Иску́сственный интелле́кт (), также ИИ, искусственный ра́зум, в самом широком смысле — комплекс инструментов, позволяющих решать задачи уровня человеческого интеллекта (такие как восприятие, обучение, рассуждение, решение проблем и принятие решений) и реализованных машинами, в частности компьютерными системами. Это направление исследований в области компьютерных наук, которая разрабатывает и изучает методы и программное обеспечение, позволяющие машинам воспринимать окружающую среду и использоват


In [26]:
t_start = time.perf_counter()
summary = summarize_text(text, file_path=FILE_PATH)
total_time = time.perf_counter() - t_start

langfuse.flush()

# Локальная сводка метрик
compression = len(text) / len(summary)
print()
print("=" * 60)
print("ИТОГОВОЕ КРАТКОЕ ИЗЛОЖЕНИЕ")
print("=" * 60)
print(summary)
print()
print("=" * 60)
print("МЕТРИКИ")
print("=" * 60)
print(f"  Исходный текст : {len(text):>8,} символов / {len(text.split()):>6,} слов")
print(f"  Краткое изложение: {len(summary):>6,} символов / {len(summary.split()):>6,} слов")
print(f"  Коэффициент сжатия: {compression} раз")
print(f"  Общее время    : {total_time:.2f} сек")

Текст разбит на 6 частей.
  Обрабатываю часть 1/6...
  Обрабатываю часть 2/6...
  Обрабатываю часть 3/6...
  Обрабатываю часть 4/6...
  Обрабатываю часть 5/6...
  Обрабатываю часть 6/6...
Финальное объединение...

ИТОГОВОЕ КРАТКОЕ ИЗЛОЖЕНИЕ
Искусственный интеллект (ИИ) — область компьютерных наук, создающая методы и программы для решения задач, требующих человеческого интеллекта. Среди инструментов ИИ — большие языковые модели, но они не равны универсальному интеллекту. ИИ применяется в поисковых системах, голосовых помощниках, автономных автомобилях, генеративных инструментах и играх. История ИИ началась в 1950-х, с периодами подъёмов и спадов; с 2012 года наблюдается новый бум благодаря глубокому обучению. В России развитие ИИ стартовало в 1960-х, с национальной стратегией с 2019 года и поддержкой государства.

В 2023–2026 годах Россия обновляет стратегию ИИ, увеличивает финансирование (около 5,2 млрд рублей в 2024), вводит стандарты и создаёт комиссию при Президенте для координации 

/var/folders/c8/dzcd_fh15mv32gjb6q_ff1gr0000gn/T/ipykernel_77930/624014699.py:139: DeprecationWarning: Trace-level input/output is deprecated. For trace attributes (user_id, session_id, tags, etc.), use propagate_attributes() instead. This method will be removed in a future major version.
  langfuse.set_current_trace_io(input=text, output=result)


In [27]:
output_path = (
    FILE_PATH.replace(".txt", "_summary.txt")
    if FILE_PATH.endswith(".txt")
    else FILE_PATH + "_summary.txt"
)

with open(output_path, "w", encoding="utf-8") as f:
    f.write(summary)

print(f"💾 Краткое изложение сохранено в: {output_path}")

💾 Краткое изложение сохранено в: sample_summary.txt


## Datasets и эксперименты

### Шаг 1. Создание датасета

In [32]:
DATASET_NAME = "summarizer-eval-v1"

dataset_items = [
    {
        "input": (
            "Искусственный интеллект (ИИ) — это раздел компьютерных наук, "
            "занимающийся созданием систем, способных выполнять задачи, "
            "которые обычно требуют человеческого интеллекта. К таким задачам "
            "относятся распознавание речи, принятие решений, перевод языков и "
            "распознавание образов. Современные системы ИИ основаны на методах "
            "машинного обучения, в частности на глубоких нейронных сетях, "
            "которые обучаются на больших объёмах данных. В последние годы ИИ "
            "стремительно развивается и находит применение в медицине, финансах, "
            "транспорте и многих других отраслях."
        ),
        "expected_output": (
            "ИИ — раздел CS для создания систем с человекоподобным интеллектом. "
            "Основан на глубоком обучении, применяется в медицине, финансах, транспорте."
        ),
    },
    {
        "input": (
            "Глобальное потепление — долгосрочное повышение средней температуры "
            "климатической системы Земли. Начиная с середины XX века оно обусловлено "
            "главным образом деятельностью человека, прежде всего сжиганием ископаемого "
            "топлива, которое приводит к накоплению парниковых газов в атмосфере. "
            "Последствия включают повышение уровня моря, учащение экстремальных "
            "погодных явлений, таяние ледников и угрозу биоразнообразию. "
            "Международное сообщество реагирует на эту угрозу через такие соглашения, "
            "как Парижское, целью которого является ограничение роста температуры "
            "в пределах 1,5–2 °C относительно доиндустриального уровня."
        ),
        "expected_output": (
            "Глобальное потепление вызвано деятельностью человека с середины XX века. "
            "Последствия: рост уровня моря, экстремальная погода, таяние льдов. "
            "Парижское соглашение нацелено на ограничение роста температуры до 1,5–2 °C."
        ),
    },
    {
        "input": (
            "Python — высокоуровневый язык программирования общего назначения, "
            "созданный Гвидо ван Россумом и впервые выпущенный в 1991 году. "
            "Философия языка делает акцент на читаемости кода и простоте синтаксиса, "
            "позволяя программистам выражать концепции в меньшем количестве строк, "
            "чем на таких языках, как C++ или Java. Python широко используется в "
            "веб-разработке, науке о данных, искусственном интеллекте, автоматизации "
            "и создании скриптов. Обширная стандартная библиотека и богатая экосистема "
            "сторонних пакетов делают его одним из самых популярных языков программирования "
            "в мире."
        ),
        "expected_output": (
            "Python — читаемый язык программирования общего назначения (с 1991). "
            "Применяется в веб-разработке, Data Science и ИИ. "
            "Популярен благодаря богатой экосистеме библиотек."
        ),
    },
    {
        "input": (
            "Квантовые вычисления — это область, использующая квантово-механические "
            "явления, такие как суперпозиция и запутанность, для выполнения вычислений. "
            "В отличие от классических битов, принимающих значения 0 или 1, квантовые "
            "биты (кубиты) могут существовать в суперпозиции обоих состояний одновременно. "
            "Это позволяет квантовым компьютерам решать определённые задачи значительно "
            "быстрее классических. Среди перспективных применений — криптография, "
            "моделирование молекул и оптимизация. Однако технология пока остаётся "
            "экспериментальной: квантовые системы требуют экстремального охлаждения "
            "и подвержены ошибкам из-за декогеренции."
        ),
        "expected_output": (
            "Квантовые компьютеры используют кубиты (суперпозиция 0 и 1), "
            "что даёт преимущество в задачах криптографии, моделирования и оптимизации. "
            "Технология экспериментальная: требует сильного охлаждения и страдает от декогеренции."
        ),
    },
]

langfuse.create_dataset(
    name=DATASET_NAME,
    description="Тестовые тексты для оценки качества суммаризации",
)

# Проверяем: если элементы уже есть — не добавляем повторно
existing = langfuse.get_dataset(DATASET_NAME)
if len(existing.items) == 0:
    for item in dataset_items:
        langfuse.create_dataset_item(
            dataset_name=DATASET_NAME,
            input={"text": item["input"]},
            expected_output={"summary": item["expected_output"]},
        )
    print(f"✅ Датасет '{DATASET_NAME}' создан, добавлено {len(dataset_items)} примера.")
else:
    print(f"ℹ️  Датасет '{DATASET_NAME}' уже содержит {len(existing.items)} элемента, пропускаем добавление.")

ℹ️  Датасет 'summarizer-eval-v1' уже содержит 4 элемента, пропускаем добавление.


### Шаг 2. Evaluator-функции

In [33]:
import json
from langfuse import Evaluation

# Кастомный evaluator: коэффициент сжатия
def compression_evaluator(*, input, output, expected_output=None, **kwargs):
    """Отношение длины изложения к длине исходного текста."""
    src = input.get("text", "") if isinstance(input, dict) else str(input)
    out = output or ""
    ratio = len(out) / len(src) if src else 0
    return Evaluation(
        name="compression_ratio",
        value=round(ratio, 3),
        comment=f"output {len(out)} символов из {len(src)} ({ratio:.1%})",
    )


# LLM-as-judge evaluator: верность фактам (faithfulness)
FAITHFULNESS_PROMPT = """\
Оцени, насколько точно краткое изложение передаёт факты и ключевые идеи оригинального текста.

Оригинал:
{source}

Краткое изложение:
{summary}

Критерии оценки (1–5):
5 — все ключевые факты сохранены, нет искажений
4 — почти все факты верны, незначительные упущения
3 — основная мысль передана, но часть деталей упущена или неточна
2 — существенные факты упущены или искажены
1 — изложение противоречит оригиналу или почти не передаёт его содержание

Ответь строго в JSON: {{"score": <1-5>, "comment": "<одно предложение>"}}
"""

def faithfulness_evaluator(*, input, output, expected_output=None, **kwargs):
    """LLM-as-judge: оценка верности фактам от 1 до 5."""
    src = input.get("text", "") if isinstance(input, dict) else str(input)
    out = output or ""
    prompt = FAITHFULNESS_PROMPT.format(source=src, summary=out)
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=150,
        )
        raw = response.choices[0].message.content.strip()
        if "```" in raw:
            raw = raw.split("```")[1].lstrip("json").strip().rstrip("```").strip()
        data = json.loads(raw)
        return Evaluation(
            name="faithfulness",
            value=float(data["score"]),
            comment=data.get("comment", ""),
        )
    except Exception as e:
        return Evaluation(name="faithfulness", value=0.0, comment=f"ошибка: {e}")


# Run-level evaluator: среднее по всему прогону
def avg_faithfulness(*, item_results, **kwargs):
    """Среднее значение faithfulness по всем элементам датасета."""
    scores = [
        ev.value
        for r in item_results
        for ev in r.evaluations
        if ev.name == "faithfulness" and ev.value is not None
    ]
    avg = sum(scores) / len(scores) if scores else 0.0
    return Evaluation(
        name="avg_faithfulness",
        value=round(avg, 3),
        comment=f"среднее по {len(scores)} элементам",
    )

### Шаг 3. Запуск эксперимента

In [34]:
# Task-функция: принимает item датасета, возвращает краткое изложение
def summarize_task(*, item, **kwargs) -> str:
    text = item.input.get("text", "")
    return summarize_text(text)


# Загружаем датасет и запускаем эксперимент
dataset = langfuse.get_dataset(DATASET_NAME)

result = langfuse.run_experiment(
    name="summarizer-eval-v1",
    description="Оценка суммаризатора: верность фактам и коэффициент сжатия",
    data=dataset.items,
    task=summarize_task,
    evaluators=[faithfulness_evaluator, compression_evaluator],
    run_evaluators=[avg_faithfulness],
    max_concurrency=2,  # ограничиваем параллельность из-за rate limits
    metadata={"model": MODEL, "chunk_size": CHUNK_SIZE},
)

langfuse.flush()
print(f"\n✅ Эксперимент завершён.")
print(f"   Обработано элементов: {len(result.item_results)}")
if result.dataset_run_url:
    print(f"   Результаты в Langfuse: {result.dataset_run_url}")

Текст помещается в один запрос.


/var/folders/c8/dzcd_fh15mv32gjb6q_ff1gr0000gn/T/ipykernel_77930/624014699.py:139: DeprecationWarning: Trace-level input/output is deprecated. For trace attributes (user_id, session_id, tags, etc.), use propagate_attributes() instead. This method will be removed in a future major version.
  langfuse.set_current_trace_io(input=text, output=result)


Текст помещается в один запрос.
Текст помещается в один запрос.
Текст помещается в один запрос.

✅ Эксперимент завершён.
   Обработано элементов: 4
   Результаты в Langfuse: http://localhost:3000/project/cmmzfphwx0006nt07b7myt2mk/datasets/cmmziq4zf000cnt07q0ogq2di/runs/265dc824-c1dc-44de-8d60-10abed9a2822


### Шаг 4. Просмотр результатов локально

In [35]:
print(f"{'№':<3} {'Faithfulness':>14} {'Compression':>13}  Изложение (первые 80 символов)")
print("-" * 80)

for i, item_result in enumerate(result.item_results, 1):
    evals = {ev.name: ev.value for ev in item_result.evaluations}
    faith = evals.get("faithfulness", "—")
    compr = evals.get("compression_ratio", "—")
    out_preview = (item_result.output or "")[:80].replace("\n", " ")
    print(f"{i:<3} {str(faith):>14} {str(compr):>13}  {out_preview}")

print()
for run_eval in result.run_evaluations:
    print(f"Run-level '{run_eval.name}': {run_eval.value}  ({run_eval.comment})")

№     Faithfulness   Compression  Изложение (первые 80 символов)
--------------------------------------------------------------------------------
1              5.0         0.776  Квантовые вычисления используют явления суперпозиции и запутанности для обработк
2              5.0         0.729  Python — высокоуровневый язык программирования общего назначения, созданный Гвид
3              5.0         0.764  Глобальное потепление — долгосрочный рост средней температуры Земли, вызванный в
4              5.0         0.709  Искусственный интеллект — раздел компьютерных наук, создающий системы для выполн

Run-level 'avg_faithfulness': 5.0  (среднее по 4 элементам)



## Дополнительные возможности Langfuse

### 1. Callback Handler (langfuse.openai)

`langfuse.openai` — drop-in замена стандартного `openai`.

In [36]:
from langfuse.openai import OpenAI as LangfuseOpenAI

lf_client = LangfuseOpenAI(
    api_key=os.getenv('OPENAI_API_KEY'),
    base_url=os.getenv('OPENAI_BASE_URL'),
    http_client=http_client,
)

# Обычный вызов — трейс создаётся автоматически, без @observe и update_current_generation
response = lf_client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": "Напиши одно предложение про Langfuse."}],
    max_tokens=60,
    # Langfuse-специфичные параметры передаются прямо сюда:
    name="callback_handler_demo",
    metadata={"demo": "callback"},
)

langfuse.flush()
print("Ответ:", response.choices[0].message.content)
print("✅ Трейс создан автоматически через callback handler — проверьте Langfuse UI")

Ответ: Langfuse — это платформа для мониторинга и анализа данных в реальном времени, помогающая компаниям улучшать качество и производительность своих приложений.
✅ Трейс создан автоматически через callback handler — проверьте Langfuse UI


### 2. User ID для трейсинга

`TraceContext` позволяет привязать трейс к конкретному пользователю, сессии и тегам.
В UI появится возможность фильтровать трейсы по `user_id`.

In [39]:
from langfuse.types import TraceContext
from langfuse import LangfuseOtelSpanAttributes
from opentelemetry import trace as otel_trace

USER_ID    = "user_42"
SESSION_ID = "session_demo_001"

with langfuse.start_as_current_observation(
    name="summarize_with_user",
    as_type="span",
    trace_context=TraceContext(
        user_id=USER_ID,
        session_id=SESSION_ID,
        tags=["summarizer", "demo"],
    ),
) as span:
    # Явно прописываем атрибуты на OTel-спане — так user_id гарантированно попадёт в трейс
    otel_span = otel_trace.get_current_span()
    otel_span.set_attribute(LangfuseOtelSpanAttributes.TRACE_USER_ID, USER_ID)
    otel_span.set_attribute(LangfuseOtelSpanAttributes.TRACE_SESSION_ID, SESSION_ID)
    otel_span.set_attribute(LangfuseOtelSpanAttributes.TRACE_TAGS, ["summarizer", "demo"])
    otel_span.set_attribute(LangfuseOtelSpanAttributes.AS_ROOT, True)

    response = client.chat.completions.create(
        model=MODEL,
        messages=[{"role": "user", "content": "Кратко: что такое трейсинг в LLM?"}],
        max_tokens=80,
    )
    result = response.choices[0].message.content.strip()
    span.update(output=result)
    trace_id = langfuse.get_current_trace_id()

langfuse.flush()
print(f"user_id  : {USER_ID}")
print(f"session  : {SESSION_ID}")
print(f"trace_id : {trace_id}")
print(f"Ответ    : {result}")
print("✅ Трейс привязан к пользователю — можно отфильтровать по User ID в Langfuse UI")

user_id  : user_42
session  : session_demo_001
trace_id : 02bbfdd47538ab23628358264291ce3e
Ответ    : Трейсинг в LLM — это процесс отслеживания и записи внутренних вычислений модели при генерации ответа. Он помогает понять, как модель пришла к конкретному выводу, анализировать последовательность операций и выявлять ошибки или улучшать интерпретируемость.
✅ Трейс привязан к пользователю — можно отфильтровать по User ID в Langfuse UI


### 3. Human Annotations

Человеческая разметка — оценки, которые ставятся вручную (не LLM).
Передаются через `create_score` с указанием `trace_id` - для примера. В реальности аннотаторы работают через UI.
Отображаются в UI рядом с автоматическими скорами.

In [40]:
human_trace_id = trace_id  # из предыдущей ячейки

# Числовая оценка: насколько хорошо изложение (1–5)
langfuse.create_score(
    trace_id=human_trace_id,
    name="human_quality",
    value=4.0,
    data_type="NUMERIC",
    comment="Хорошее изложение, все ключевые факты сохранены",
)

# Категориальная оценка: пригодно ли для публикации
langfuse.create_score(
    trace_id=human_trace_id,
    name="human_ready_to_publish",
    value="yes",
    data_type="CATEGORICAL",
    comment="Можно публиковать без правок",
)

# Булева оценка: есть ли галлюцинации
langfuse.create_score(
    trace_id=human_trace_id,
    name="human_has_hallucination",
    value=False,
    data_type="BOOLEAN",
    comment="Галлюцинаций не обнаружено",
)

langfuse.flush()
print(f"✅ Human annotations добавлены к трейсу {human_trace_id}")
print("   Скоры видны в Langfuse UI → Traces → выберите трейс → вкладка Scores")

✅ Human annotations добавлены к трейсу 02bbfdd47538ab23628358264291ce3e
   Скоры видны в Langfuse UI → Traces → выберите трейс → вкладка Scores


### 4. LLM-as-a-judge

Паттерн прямого применения к конкретному трейсу (не через `run_experiment`).

In [41]:
CONCISENESS_PROMPT = """\
Оцени краткость изложения относительно оригинального текста.

Оригинал ({src_words} слов):
{source}

Изложение ({out_words} слов):
{summary}

Критерии краткости (1–5):
5 — очень кратко, только самое главное, без воды
4 — кратко, минимум лишних деталей
3 — умеренно кратко, есть небольшие излишества
2 — слишком подробно для краткого изложения
1 — почти не сжато, объём сравним с оригиналом

Ответь строго в JSON: {{"score": <1-5>, "comment": "<одно предложение>"}}
"""

def conciseness_judge(source: str, summary: str, trace_id: str) -> dict:
    """LLM-as-judge: оценка краткости, результат записывается в трейс."""
    prompt = CONCISENESS_PROMPT.format(
        source=source,
        summary=summary,
        src_words=len(source.split()),
        out_words=len(summary.split()),
    )
    try:
        response = client.chat.completions.create(
            model=MODEL,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.0,
            max_tokens=150,
        )
        raw = response.choices[0].message.content.strip()
        if "```" in raw:
            raw = raw.split("```")[1].lstrip("json").strip().rstrip("```").strip()
        data = json.loads(raw)
        langfuse.create_score(
            trace_id=trace_id,
            name="llm_conciseness",
            value=float(data["score"]),
            data_type="NUMERIC",
            comment=data.get("comment", ""),
        )
        return data
    except Exception as e:
        print(f"Ошибка judge: {e}")
        return {}


# Применяем к трейсу из суммаризации (summary и text из ячеек выше)
result_data = conciseness_judge(
    source=text,
    summary=summary,
    trace_id=human_trace_id,
)

langfuse.flush()
print(f"llm_conciseness score : {result_data.get('score')}")
print(f"Комментарий           : {result_data.get('comment')}")

llm_conciseness score : 4
Комментарий           : Изложение хорошо сокращает оригинал, сохраняя ключевые темы и факты, но содержит некоторые детали, которые можно было бы опустить для ещё большей краткости.


## Анализ результатов через Langfuse UI

### Трейсы и spans

На скриншоте раздела **Tracing → Traces** видны все запуски суммаризатора:

- Каждая строка — один вызов `summarize_pipeline` (корневой трейс)
- Колонка **Input** содержит исходный текст, **Output** — итоговое изложение
- Трейс `summarize_with_user` привязан к `user_id = user_42` — виден в фильтре **User ID** на боковой панели
- Трейс `callback_handler_demo` создан автоматически через `langfuse.openai` без `@observe`

Раздел **Tracing → Observations** показывает все дочерние spans и generations:

- `chunk_0`, `chunk_1`, ... — отдельные LLM-вызовы (тип `generation`)
- `chunk_done`, `chunks_ready`, `file_loaded` — events (точки без длительности)
- Колонка **Latency** показывает время каждого вызова: от 1.8с до 13.7с
- Колонка **Total** — стоимость каждого observation

### Анализ времени ответа

Из таблицы Observations (скриншот `03-47-57`):

| Observation | Latency | Вывод |
|---|---|---|
| `chunk_1`–`chunk_5` | 8–14 сек | основной текст (крупные чанки) |
| `chunk_0` (финал) | 4–7 сек | объединение частичных изложений |
| `callback_handler_demo` | 1.8 сек | короткий тестовый запрос |
| `summarize_with_user` | 2.1 сек | короткий вопрос |

Время пропорционально длине чанка — чем больше токенов, тем дольше ответ модели.

### Анализ стоимости

Dashboard (скриншот `03-49-36`) за 1 день:
- **32 трейса**, **155 observations**, **46 числовых скоров**, **1 категориальный скор**

Из таблицы Observations: стоимость одного чанка ~$0.00026–$0.00093 в зависимости от объёма текста. Для gpt-4.1-mini при `CHUNK_SIZE=12000` символов весь документ (~63k символов) обходится менее **$0.01**.

### Результаты эксперимента с датасетом

Скриншот `03-52-31` — раздел **Datasets → summarizer-eval-v1 → Runs**:

| Текст | Latency | `compression_ratio` | `faithfulness` |
|---|---|---|---|
| Искусственный интеллект | 14.3с | 0.709 | 5.0 |
| Глобальное потепление | 15.4с | 0.764 | 5.0 |
| Квантовые вычисления | 4.4с | 0.776 | 5.0 |
| Python | 9.4с | 0.729 | 5.0 |

**Выводы:**
- Faithfulness = 5.0 по всем примерам — модель точно передаёт факты оригинала
- Compression ratio 0.71–0.78 означает, что изложение составляет ~75% длины исходного текста для коротких текстов датасета (они помещаются в один чанк без агрессивного сжатия)
- Для длинного файла `sample.txt` (63k символов) compression ratio составил 5.4% — многочанковая обработка даёт значительно лучшее сжатие
- Latency коррелирует с длиной текста: квантовые вычисления (самый короткий) — 4.4с, потепление (самый длинный) — 15.4с